# Appendice A: Grafo Computazionale, Autograd e Training Loop in PyTorch
Questo notebook unisce tutti i concetti fondamentali studiati nell'Appendice A del libro per costruire, addestrare e testare una rete neurale in PyTorch.
In questa prima cella andremo a importare le librerie necessarie e a configurare il dispositivo di calcolo. Poiché stai lavorando su un Mac moderno, configureremo PyTorch per usare MPS (Metal Performance Shaders), che permette di sfruttare la GPU integrata nei chip Apple Silicon velocizzando drasticamente i calcoli rispetto alla sola CPU.

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Configurazione del dispositivo per Mac (Apple Silicon)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("GPU attivata")
else:
    device = torch.device("cpu")
    print("MPS non disponibile, utilizzo della CPU.")

GPU attivata


# 1. Definizione del Dataset e del DataLoader
Per addestrare la rete in modo efficiente senza saturare la memoria, dobbiamo strutturare i dati. Creeremo:

Un Dataset personalizzato ereditando da torch.utils.data.Dataset. Questo definirà la sorgente dei nostri dati (in questo caso, coordinate bidimensionali fittizie).
Un DataLoader che si occuperà di raggruppare i dati in piccoli gruppi chiamati batch e di rimescolarli (shuffle=True) ad ogni epoca per migliorare l'apprendimento.

In [11]:
# Creiamo un dataset fittizio bidimensionale
class ToyClassificationDataset(Dataset):
    def __init__(self):
        # Definiamo 6 punti nello spazio bidimensionale (X, Y)
        # I primi 3 appartengono alla classe 0, gli ultimi 3 alla classe 1
        self.features = torch.tensor([
            [-1.5, 2.0],
            [-2.0, 1.5],
            [-1.8, 2.2],
            [1.5, -2.0],
            [2.0, -1.5],
            [1.8, -2.2]
        ], dtype=torch.float32)
        
        # Etichette delle classi (target)
        self.labels = torch.tensor([0, 0, 0, 1, 1, 1], dtype=torch.long)

    def __len__(self):
        # Restituisce il numero totale di elementi
        return len(self.labels)

    def __getitem__(self, index):
        # Restituisce un singolo esempio (coppia feature-etichetta) dato un indice
        return self.features[index], self.labels[index]

# Inizializziamo il dataset e il dataloader con batch_size di 2 elementi
dataset = ToyClassificationDataset()
train_loader = DataLoader(dataset, batch_size=2, shuffle=True)

print(f"Dataset creato con successo! Numero di campioni: {len(dataset)}")

Dataset creato con successo! Numero di campioni: 6


## 2. Creazione della Rete Neurale Multistrato (Capitolo A.5)
Costruiamo ora una rete neurale ereditando da nn.Module.
Per evitare il collasso lineare (la sovrapposizione di sole funzioni lineari che si ridurrebbe a un'unica regressione piatta), introduciamo una funzione di attivazione non-lineare ReLU tra i due strati. Questo permette alla rete di curvare lo spazio geometrico dei dati e trovare una frontiera di decisione efficace.

In [12]:
class ToyNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        # Usiamo nn.Sequential per incatenare i layer linearmente
        self.layers = nn.Sequential(
            # Strato d'ingresso: prende le 2 coordinate e le mappa in 4 neuroni intermedi
            nn.Linear(in_features=2, out_features=4),
            
            # Non-linearità che "piega lo spazio"
            nn.ReLU(),
            
            # Strato di output: dai 4 neuroni intermedi ricava 2 punteggi (logits)
            # Un logit per la classe 0 e un logit per la classe 1
            nn.Linear(in_features=4, out_features=2)
        )
        
    def forward(self, x):
        # Definiamo il percorso dei dati
        return self.layers(x)

# Istanziamo il modello e lo spostiamo sulla GPU del Mac (device mps o cpu)
model = ToyNetwork().to(device)
print(model)

ToyNetwork(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=4, bias=True)
    (1): ReLU()
    (2): Linear(in_features=4, out_features=2, bias=True)
  )
)


## 3. Il Training Loop: Unire Forward, Backward e Ottimizzazione (Capitoli A.3, A.4 e A.7)
In questa fase uniamo tutti i processi nel Training Loop:

Forward Propagation: I dati del batch passano nella rete per calcolare i logits. Viene poi calcolata la Loss (entropia incrociata), che confronta la predizione con le etichette reali.
Reset dei gradienti (optimizer.zero_grad()): Azzeriamo i gradienti accumulati nel passo precedente per evitare interferenze.
Backward Propagation (loss.backward()): Il motore Autograd di PyTorch risale il grafo computazionale a ritroso applicando la regola della catena (Chain Rule) e calcolando i gradienti per ogni parametro.
Aggiornamento dei pesi (optimizer.step()): L'ottimizzatore applica la discesa del gradiente aggiornando numericamente i pesi del modello.

In [13]:
# Fissiamo il seed per la riproducibilità
torch.manual_seed(42)

# Definiamo l'ottimizzatore (Stochastic Gradient Descent) con un tasso di apprendimento (lr)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

print("=== INIZIO DEL TRAINING LOOP ===")

# Eseguiamo l'addestramento per 10 epoche
for epoch in range(10):
    model.train() # Impostiamo il modello in modalità addestramento
    epoch_loss = 0.0
    
    for batch_idx, (features, labels) in enumerate(train_loader):
        # Spostiamo i dati sul dispositivo (MPS) prima del calcolo
        features = features.to(device)
        labels = labels.to(device)
        
        # 1. Forward Pass
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        
        # 2. Reset dei gradienti
        optimizer.zero_grad()
        
        # 3. Backward Pass
        loss.backward()
        
        # 4. Aggiornamento dei pesi
        optimizer.step()
        
        epoch_loss += loss.item()
        
    # Calcoliamo la loss media dell'epoca corrente
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoca {epoch+1:2d}/10 | Loss media: {avg_loss:.4f}")



=== INIZIO DEL TRAINING LOOP ===
Epoca  1/10 | Loss media: 0.6740
Epoca  2/10 | Loss media: 0.5045
Epoca  3/10 | Loss media: 0.4169
Epoca  4/10 | Loss media: 0.3713
Epoca  5/10 | Loss media: 0.3299
Epoca  6/10 | Loss media: 0.3007
Epoca  7/10 | Loss media: 0.2753
Epoca  8/10 | Loss media: 0.2529
Epoca  9/10 | Loss media: 0.2332
Epoca 10/10 | Loss media: 0.2157


## 4. Verifica Finale e Predizione
Ora che la rete è stata addestrata e l'ottimizzatore ha guidato i parametri verso il punto di minimo errore sul grafo computazionale, verifichiamo se il modello è in grado di classificare correttamente i nostri dati di partenza.

In [ ]:
# Impostiamo il modello in modalità valutazione (disattiva dropout ecc.)
model.eval()

# Usiamo torch.no_grad() per evitare che PyTorch costruisca inutilmente il grafo computazionale
# durante la fase di test, risparmiando memoria ed energia
with torch.no_grad():
    # Spostiamo l'intero dataset sul dispositivo per fare la verifica finale
    all_features = dataset.features.to(device)
    all_labels = dataset.labels.tolist()
    
    # Eseguiamo il forward pass
    logits_pred = model(all_features)
    
    # Convertiamo i logits in probabilità usando softmax ed estraiamo l'indice della classe massima
    predictions = torch.argmax(logits_pred, dim=1).cpu().tolist()

print("=== CONFRONTO FINALE ===")
print("Etichette Reali:     ", all_labels)
print("Etichette Predette:  ", predictions)

if all_labels == predictions:
    print("\nSuccesso! La rete ha imparato a separare perfettamente le due classi!")
else:
    print("\nLa rete necessita di ulteriori epoche di addestramento o di un learning rate differente.")

=== CONFRONTO FINALE ===
Etichette Reali:      [0, 0, 0, 1, 1, 1]
Etichette Predette:   [0, 0, 0, 1, 1, 1]

🎉 Successo! La rete ha imparato a separare perfettamente le due classi!


# 5. Salvare e Caricare i Modelli in PyTorch (Capitolo A.8)
In PyTorch, il modo raccomandato per salvare un modello è salvare il suo state_dict.
Cos'è lo state_dict?
Lo state_dict è semplicemente un dizionario standard di Python che mappa ciascun layer della rete ai suoi rispettivi parametri addestrabili, ovvero le matrici di pesi (weights) e i vettori di bias.
Quando salviamo il modello, esportiamo questo dizionario in un file con estensione .pth o .pt (le convenzioni standard di PyTorch).
Per caricare il modello in un secondo momento, dobbiamo compiere tre passaggi logici:

Creare un'istanza vuota del modello (la struttura del codice deve essere identica a quella originale, altrimenti le matrici non coincideranno).
Leggere il dizionario da disco usando torch.load.
Iniettare i pesi caricati all'interno dell'istanza del modello tramite .load_state_dict().

Eseguiamo la cella successiva per salvare i pesi del modello che abbiamo addestrato poco fa.

In [18]:
# 1. SALVATAGGIO DEI PESI (Solo modello)
# Salviamo lo state_dict (il dizionario dei pesi) in un file chiamato 'modello_toy.pth'
nome_file = "modello_toy.pth"
torch.save(model.state_dict(), nome_file)
print(f"Pesi del modello salvati con successo in: '{nome_file}'")

# Visualizziamo un'anteprima dello state_dict per capire cosa contiene realmente
print("\nChiavi e forme dei tensori registrati nello state_dict:")
for layer_name, weights in model.state_dict().items():
    print(f"Layer: {layer_name:<25} | Forma del tensore: {list(weights.shape)}")

Pesi del modello salvati con successo in: 'modello_toy.pth'

Chiavi e forme dei tensori registrati nello state_dict:
Layer: layers.0.weight           | Forma del tensore: [4, 2]
Layer: layers.0.bias             | Forma del tensore: [4]
Layer: layers.2.weight           | Forma del tensore: [2, 4]
Layer: layers.2.bias             | Forma del tensore: [2]


# 6. Ripristinare il modello da disco
Ora simuliamo di essere in una nuova sessione di lavoro:

Creiamo un modello completamente nuovo e "vergine" (inizializzato con pesi casuali).
Verifichiamo che le sue prestazioni siano scarse (vicine a un tiraggio di dado casuale).
Carichiamo i pesi salvati in precedenza.
Verifichiamo che il modello ripristinato sia tornato a essere preciso al 100%.

Nota per Mac: Quando si carica un modello in PyTorch su un computer diverso o su un dispositivo diverso (ad esempio, se è stato salvato su GPU e vuoi caricarlo su CPU o viceversa), è caldamente raccomandato impostare il parametro map_location all'interno di torch.load per mappare correttamente i tensori sul dispositivo di destinazione (nel nostro caso, device che punta a mps o cpu)

In [19]:
# 1. Istanziamo un nuovo modello identico (ma con pesi casuali diversi)
modello_nuovo = ToyNetwork().to(device)
modello_nuovo.eval()

# Facciamo una predizione di prova con il modello non addestrato
with torch.no_grad():
    all_features = dataset.features.to(device)
    all_labels = dataset.labels.tolist()
    logits_casuali = modello_nuovo(all_features)
    pred_casuali = torch.argmax(logits_casuali, dim=1).cpu().tolist()

print("=== VERIFICA MODELLO CASUALE (NON ADDESTRATO) ===")
print("Etichette Reali:     ", all_labels)
print("Etichette Predette:  ", pred_casuali)
print(f"La predizione è corretta? {'Sì' if all_labels == pred_casuali else 'No'}")


# 2. Carichiamo lo stato salvato nel file
pesi_caricati = torch.load("modello_toy.pth", map_location=device)

# 3. Applichiamo i pesi al modello nuovo
modello_nuovo.load_state_dict(pesi_caricati)
modello_nuovo.eval()

# Ripetiamo la predizione con il modello ripristinato
with torch.no_grad():
    logits_ripristinati = modello_nuovo(all_features)
    pred_ripristinate = torch.argmax(logits_ripristinati, dim=1).cpu().tolist()

print("\n=== VERIFICA MODELLO DOPO IL CARICAMENTO DEI PESI ===")
print("Etichette Reali:     ", all_labels)
print("Etichette Predette:  ", pred_ripristinate)
print(f"La predizione è corretta? {'Sì, al 100%!' if all_labels == pred_ripristinate else 'No'}")

=== VERIFICA MODELLO CASUALE (NON ADDESTRATO) ===
Etichette Reali:      [0, 0, 0, 1, 1, 1]
Etichette Predette:   [0, 0, 0, 1, 1, 1]
La predizione è corretta? Sì

=== VERIFICA MODELLO DOPO IL CARICAMENTO DEI PESI ===
Etichette Reali:      [0, 0, 0, 1, 1, 1]
Etichette Predette:   [0, 0, 0, 1, 1, 1]
La predizione è corretta? Sì, al 100%!


/var/folders/yw/h50w2x2s4vlg3b39v4071rnw0000gn/T/ipykernel_35709/2704604504.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pesi_caricati = torch.load("modello_toy.pth"

# 7. Salvataggio Avanzato per Continuare l'Addestramento (Capitolo 5.4)
Se stai addestrando un modello linguistico di grandi dimensioni (come un Transformer), l'addestramento può richiedere ore, giorni o settimane. Non puoi fare tutto in un'unica sessione di calcolo.
Il problema: Se salvi solo lo state_dict del modello e poi decidi di riprendere l'addestramento, l'ottimizzatore (specialmente se usi algoritmi adattivi come AdamW) verrà azzerato. AdamW tiene traccia della media mobile dei gradienti passati per calcolare la traiettoria migliore (il "momento" o inerzia). Senza queste informazioni storiche memorizzate nell'ottimizzatore, l'addestramento potrebbe perdere stabilità o fallire la convergenza quando decidi di ripartire.
La soluzione: Salvare un Checkpoint complessivo. Invece di salvare solo lo state_dict del modello, salviamo un dizionario di Python che contiene:

Lo state_dict del modello.
Lo state_dict dell'ottimizzatore.
Eventuali metadati utili, come il numero di epoche completate o l'ultima Loss registrata.

In questo modo, possiamo ripristinare sia i pesi della rete sia l'inerzia dell'ottimizzatore, riprendendo l'addestramento esattamente da dove lo avevamo interrotto.

In [20]:
# Creiamo un ottimizzatore d'esempio (AdamW) collegato al nostro modello
optimizer_adam = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.1)

# Definiamo i dati da salvare nel checkpoint
checkpoint_salvato = {
    "epoca_corrente": 10,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer_adam.state_dict(),
    "last_loss": 0.015
}

# Salviamo il file checkpoint sul disco
torch.save(checkpoint_salvato, "checkpoint_completo.pth")
print("Checkpoint completo (modello + ottimizzatore + epoca) salvato con successo!")


# --- FARE IL RIPRISTINO (Codice di caricamento) ---

# 1. Carichiamo il file checkpoint complessivo
checkpoint_caricato = torch.load("checkpoint_completo.pth", map_location=device)

# 2. Inizializziamo il modello e l'ottimizzatore vuoti
modello_ripartenza = ToyNetwork().to(device)
ottimizzatore_ripartenza = torch.optim.AdamW(modello_ripartenza.parameters(), lr=0.001, weight_decay=0.1)

# 3. Ripristiniamo gli stati interni di entrambi
modello_ripartenza.load_state_dict(checkpoint_caricato["model_state_dict"])
ottimizzatore_ripartenza.load_state_dict(checkpoint_caricato["optimizer_state_dict"])

# 4. Estraiamo i metadati salvati
epoca_da_cui_ripartire = checkpoint_caricato["epoca_corrente"]
ultima_loss = checkpoint_caricato["last_loss"]

print(f"\nStato ripristinato! Pronto a ripartire dall'epoca {epoca_da_cui_ripartire + 1} (Ultima loss registrata: {ultima_loss})")

Checkpoint completo (modello + ottimizzatore + epoca) salvato con successo!

Stato ripristinato! Pronto a ripartire dall'epoca 11 (Ultima loss registrata: 0.015)


/var/folders/yw/h50w2x2s4vlg3b39v4071rnw0000gn/T/ipykernel_35709/1267552898.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_caricato = torch.load("checkpoint